## Compare HMM vs CHMM Distillation

This notebook compares the trained baseline HMM against one or more saved CHMM checkpoints on the same held-out data.

It answers three questions:

1. Which saved CHMM checkpoint is currently best on the dev set?
2. How does the best CHMM compare to the HMM on next-token prediction?
3. How do their free generations look side by side?


In [1]:
from pathlib import Path
import math
import random
import sys

import torch
import pandas as pd
from transformers import AutoTokenizer

# Make local modules importable when the notebook is opened from distillation/.
ROOT = Path.cwd().resolve()
if ROOT.name == 'distillation':
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT.parent))
else:
    sys.path.insert(0, str(ROOT / 'distillation'))
    sys.path.insert(0, str(ROOT))

from ctrlg import HMM
from chmm import CHMM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_MODEL_PATH = 'gpt2-large'
DATASET = 'gpt2-large'
DATA_DIR = Path('./workspace/hmm_data') / DATASET
DEV_FILE = DATA_DIR / f'{DATASET}.dev'

HMM_MODEL_DIR = Path('./workspace/models/hmm_gpt2-large_4096')
HMM_CHECKPOINT = 200

CHMM_MODEL_DIR = Path('./workspace/models/chmm_gpt2-large')
CHMM_CHECKPOINT = 300  # int, 'latest', or 'best_saved'

EVAL_BATCH_SIZE = 512
DEV_EVAL_LIMIT = 5000
PREFIX_EVAL_COUNT = 32
PREFIX_LEN = 8
TOPK = 10
GEN_MAX_NEW_TOKENS = 32
NUM_GENERATIONS = 5
SEED = 13

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
EOS_TOKEN_ID = tokenizer.eos_token_id
VOCAB_SIZE = tokenizer.vocab_size

print(f'device={DEVICE}')
print(f'dev_file={DEV_FILE}')
print(f'hmm_model_dir={HMM_MODEL_DIR}')
print(f'chmm_model_dir={CHMM_MODEL_DIR}')
print(f'chmm_checkpoint_mode={CHMM_CHECKPOINT}')


device=cuda
dev_file=workspace/hmm_data/gpt2-large/gpt2-large.dev
hmm_model_dir=workspace/models/hmm_gpt2-large_4096
chmm_model_dir=workspace/models/chmm_gpt2-large
chmm_checkpoint_mode=300


In [2]:
def checkpoint_number(path: Path) -> int:
    return int(path.name.split('-')[-1])

def list_checkpoints(model_dir: Path):
    return sorted([p for p in model_dir.glob('checkpoint-*') if p.is_dir()], key=checkpoint_number)

def resolve_checkpoint(model_dir: Path, selector):
    checkpoints = list_checkpoints(model_dir)
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoints found in {model_dir}')
    if isinstance(selector, int):
        path = model_dir / f'checkpoint-{selector}'
        if not path.exists():
            raise FileNotFoundError(f'Checkpoint not found: {path}')
        return path
    if selector == 'latest':
        return checkpoints[-1]
    if selector == 'best_saved':
        return None
    raise ValueError(f'Unsupported checkpoint selector: {selector}')

torch.manual_seed(SEED)
random.seed(SEED)

hmm_ckpt = resolve_checkpoint(HMM_MODEL_DIR, HMM_CHECKPOINT)
hmm_model = HMM.from_pretrained(hmm_ckpt, map_location='cpu').to(DEVICE)
hmm_model.eval()

print(f'hmm_ckpt={hmm_ckpt}')
print(f'HMM hidden_states={hmm_model.hidden_states}, vocab_size={hmm_model.vocab_size}')


Loading weights from local directory
hmm_ckpt=workspace/models/hmm_gpt2-large_4096/checkpoint-200
HMM hidden_states=4096, vocab_size=50257


In [3]:
def trim_lengths(input_ids: torch.Tensor, eos_token_id: int) -> torch.Tensor:
    eos_mask = input_ids.eq(eos_token_id)
    has_eos = eos_mask.any(dim=1)
    first_eos = torch.argmax(eos_mask.int(), dim=1)
    full_length = torch.full_like(first_eos, input_ids.shape[1] - 1)
    first_eos = torch.where(has_eos, first_eos, full_length)
    return first_eos + 1

def iter_length_buckets(input_ids: torch.Tensor, eos_token_id: int):
    lengths = trim_lengths(input_ids, eos_token_id)
    for length in torch.unique(lengths, sorted=True).tolist():
        bucket = input_ids[lengths == length, :length]
        if bucket.numel() > 0:
            yield int(length), bucket.contiguous()

def score_model_trimmed(model, sequences: torch.Tensor, batch_size: int, eos_token_id: int) -> dict:
    total_ll = 0.0
    total_sequences = 0
    total_tokens = 0
    with torch.no_grad():
        for start in range(0, sequences.shape[0], batch_size):
            batch = sequences[start:start + batch_size]
            for _, bucket in iter_length_buckets(batch, eos_token_id):
                ll = model.loglikelihood(bucket, batch_size=min(batch_size, len(bucket)))
                total_ll += float(ll.item())
                total_sequences += int(bucket.shape[0])
                total_tokens += int(bucket.numel())
    nll_tok = -total_ll / max(total_tokens, 1)
    return {
        'total_loglik': total_ll,
        'num_sequences': total_sequences,
        'num_tokens': total_tokens,
        'nll_per_token_nats': nll_tok,
        'nll_per_token_bits': nll_tok / math.log(2.0),
        'perplexity': math.exp(nll_tok),
    }

def hmm_filter_state(prefix_ids, model):
    device = model.alpha_exp.device
    prefix = torch.tensor(prefix_ids, dtype=torch.long, device=device)

    transition = model.alpha_exp
    emission = model.beta.exp()
    state = torch.softmax(model.gamma, dim=0) * emission[:, prefix[0]]
    state = state / state.sum().clamp_min(1e-12)

    for token_id in prefix[1:].tolist():
        state = torch.matmul(state, transition)
        state = state * emission[:, token_id]
        state = state / state.sum().clamp_min(1e-12)

    return state

def hmm_next_token_probs(prefix_ids, model):
    state = hmm_filter_state(prefix_ids, model)
    transition = model.alpha_exp
    emission = model.beta.exp()
    next_state = torch.matmul(state, transition)
    probs = torch.matmul(next_state, emission)
    return probs / probs.sum().clamp_min(1e-12)

def chmm_filter_state(prefix_ids, model):
    tokens = torch.tensor(prefix_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    _, cache = model.forward_backward(tokens, return_messages=True)
    alpha, beta, state_ids, state_mask, _ = cache

    logits = alpha[-1, 0] + beta[-1, 0]
    mask = state_mask[0, -1]
    logits = logits.masked_fill(~mask, float('-inf'))
    logits = logits - torch.logsumexp(logits, dim=-1, keepdim=True)

    active_state_ids = state_ids[0, -1][mask]
    active_state_probs = logits.exp()[mask]
    return active_state_ids, active_state_probs

def chmm_next_token_probs(prefix_ids, model):
    active_state_ids, active_state_probs = chmm_filter_state(prefix_ids, model)
    token_probs = model.token_log_probs[active_state_ids].float().exp()
    probs = torch.sum(active_state_probs[:, None] * token_probs, dim=0)
    return probs / probs.sum().clamp_min(1e-12)

def topk_summary(probs: torch.Tensor, tokenizer, k: int = 10):
    top = torch.topk(probs, k=k)
    rows = []
    for rank, (token_id, prob) in enumerate(zip(top.indices.tolist(), top.values.tolist()), start=1):
        rows.append({
            'rank': rank,
            'token_id': token_id,
            'token_str': repr(tokenizer.decode([token_id])),
            'prob': prob,
        })
    return pd.DataFrame(rows)

def js_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-12) -> float:
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)
    m = 0.5 * (p + q)
    kl_pm = torch.sum(p * (p.log() - m.log()))
    kl_qm = torch.sum(q * (q.log() - m.log()))
    return float(0.5 * (kl_pm + kl_qm))

def model_device(model):
    if hasattr(model, "device"):
        return torch.device(model.device)
    try:
        return next(model.parameters()).device
    except (StopIteration, AttributeError, TypeError):
        return torch.device(DEVICE)

def sample_autoregressive(next_token_fn, model, prefix_ids, max_new_tokens: int, eos_token_id: int, seed: int):
    tokens = list(prefix_ids)
    gen = torch.Generator(device=model_device(model)).manual_seed(seed)
    for _ in range(max_new_tokens):
        probs = next_token_fn(tokens, model)
        next_tok = torch.multinomial(probs, num_samples=1, generator=gen).item()
        tokens.append(next_tok)
        if next_tok == eos_token_id:
            break
    return tokens


In [4]:
dev_sequences = torch.load(DEV_FILE, map_location='cpu', weights_only=True).long()
if DEV_EVAL_LIMIT is not None:
    dev_sequences = dev_sequences[:DEV_EVAL_LIMIT]

hmm_metrics = score_model_trimmed(hmm_model, dev_sequences, EVAL_BATCH_SIZE, EOS_TOKEN_ID)

chmm_rows = []
for ckpt_dir in list_checkpoints(CHMM_MODEL_DIR):
    chmm_model_tmp = CHMM.from_pretrained(ckpt_dir, map_location=DEVICE).to(DEVICE)
    chmm_model_tmp.eval()
    metrics = score_model_trimmed(chmm_model_tmp, dev_sequences, EVAL_BATCH_SIZE, EOS_TOKEN_ID)
    chmm_rows.append({
        'checkpoint': checkpoint_number(ckpt_dir),
        'path': str(ckpt_dir),
        **metrics,
    })
    del chmm_model_tmp
    if DEVICE.startswith('cuda'):
        torch.cuda.empty_cache()

chmm_metrics_df = pd.DataFrame(chmm_rows).sort_values('checkpoint').reset_index(drop=True)
best_row = chmm_metrics_df.sort_values('nll_per_token_nats').iloc[0]

if CHMM_CHECKPOINT == "best_saved":
    selected_chmm_ckpt = Path(best_row['path'])
elif CHMM_CHECKPOINT == "latest":
    selected_chmm_ckpt = resolve_checkpoint(CHMM_MODEL_DIR, "latest")
else:
    selected_chmm_ckpt = resolve_checkpoint(CHMM_MODEL_DIR, CHMM_CHECKPOINT)

selected_checkpoint_number = checkpoint_number(selected_chmm_ckpt)
selected_chmm_metrics = chmm_metrics_df[chmm_metrics_df['checkpoint'] == selected_checkpoint_number].iloc[0].to_dict()

comparison_df = pd.DataFrame([
    {'model': 'HMM', **hmm_metrics},
    {'model': f'CHMM@{selected_checkpoint_number}', **selected_chmm_metrics},
]).set_index('model')

print(f'selected_chmm_ckpt={selected_chmm_ckpt}')
print('All saved CHMM checkpoints:')
display(chmm_metrics_df)
comparison_df


selected_chmm_ckpt=workspace/models/chmm_gpt2-large/checkpoint-300
All saved CHMM checkpoints:


,checkpoint,path,total_loglik,num_sequences,num_tokens,nll_per_token_nats,nll_per_token_bits,perplexity
0,0,workspace/models/chmm_gpt2-large/checkpoint-0,-795892.623212,5000,157981,5.037901,7.268155,154.146108
1,50,workspace/models/chmm_gpt2-large/checkpoint-50,-769752.408460,5000,157981,4.872437,7.029440,130.638846
2,100,workspace/models/chmm_gpt2-large/checkpoint-100,-757746.597920,5000,157981,4.796441,6.919802,121.078771
3,150,workspace/models/chmm_gpt2-large/checkpoint-150,-756880.053908,5000,157981,4.790956,6.911889,120.416458
4,200,workspace/models/chmm_gpt2-large/checkpoint-200,-754637.406286,5000,157981,4.776761,6.891409,118.719140
5,250,workspace/models/chmm_gpt2-large/checkpoint-250,-754370.017113,5000,157981,4.775068,6.888967,118.518373
6,300,workspace/models/chmm_gpt2-large/checkpoint-300,-753243.169950,5000,157981,4.767935,6.878676,117.676013


,total_loglik,num_sequences,num_tokens,nll_per_token_nats,nll_per_token_bits,perplexity,checkpoint,path
model,,,,,,,,
HMM,-701919.845877,5000,157981,4.443065,6.409988,85.035172,NaN,NaN
CHMM@300,-753243.169950,5000,157981,4.767935,6.878676,117.676013,300.0,workspace/models/chmm_gpt2-large/checkpoint-300


In [5]:
chmm_model = CHMM.from_pretrained(selected_chmm_ckpt, map_location=DEVICE).to(DEVICE)
chmm_model.eval()
print(f'CHMM hidden_states={chmm_model.hidden_states}, vocab_size={chmm_model.vocab_size}, max_clones={chmm_model.max_clones}')


CHMM hidden_states=55889, vocab_size=50257, max_clones=4


In [6]:
prefix_rows = []
usable = dev_sequences[trim_lengths(dev_sequences, EOS_TOKEN_ID) > (PREFIX_LEN + 1)]
sample_indices = list(range(min(PREFIX_EVAL_COUNT, len(usable))))

for example_id in sample_indices:
    seq = usable[example_id]
    prefix = seq[:PREFIX_LEN].tolist()
    target_next = int(seq[PREFIX_LEN].item())

    hmm_probs = hmm_next_token_probs(prefix, hmm_model).cpu()
    chmm_probs = chmm_next_token_probs(prefix, chmm_model).cpu()

    hmm_topk = torch.topk(hmm_probs, k=TOPK).indices.tolist()
    chmm_topk = torch.topk(chmm_probs, k=TOPK).indices.tolist()
    overlap = len(set(hmm_topk) & set(chmm_topk))

    hmm_rank = int((torch.argsort(hmm_probs, descending=True) == target_next).nonzero(as_tuple=False)[0].item() + 1)
    chmm_rank = int((torch.argsort(chmm_probs, descending=True) == target_next).nonzero(as_tuple=False)[0].item() + 1)

    prefix_rows.append({
        'example_id': example_id,
        'prefix_text': tokenizer.decode(prefix),
        'target_next': repr(tokenizer.decode([target_next])),
        'hmm_target_prob': float(hmm_probs[target_next].item()),
        'chmm_target_prob': float(chmm_probs[target_next].item()),
        'hmm_target_rank': hmm_rank,
        'chmm_target_rank': chmm_rank,
        'top10_overlap': overlap,
        'js_divergence': js_divergence(hmm_probs, chmm_probs),
    })

prefix_df = pd.DataFrame(prefix_rows)
prefix_df.head(10)


,example_id,prefix_text,target_next,hmm_target_prob,chmm_target_prob,hmm_target_rank,chmm_target_rank,top10_overlap,js_divergence
0,0,Mark Olshansky and Alex Sand,'ro',0.002750,0.009816,60,22,2,0.446084
1,1,하고 말,'�',0.315399,0.054641,1,2,3,0.432893
2,2,• • •,' ',0.261660,0.149247,1,1,5,0.187533
3,3,***************__ *************** //__________...,'________',0.462818,0.282219,1,1,8,0.080556
4,4,༼ つ ◕_�,'�',0.989725,0.963216,1,1,4,0.013770
5,5,______________________________________________...,'C',0.000598,0.000942,140,98,6,0.157996
6,6,�任玉的�,'�',0.017097,0.152204,23,3,3,0.413964
7,7,iay 0 yololo 10,' w',0.001270,0.000125,88,645,7,0.189929
8,8,an alarming study has taken place.,' The',0.034739,0.028350,6,5,8,0.123083
9,9,______________________________________________...,' residents',0.000020,0.000007,2551,7096,8,0.099467


In [7]:
example_id = 0
seq = usable[example_id]
prefix = seq[:PREFIX_LEN].tolist()
target_next = int(seq[PREFIX_LEN].item())

print('PREFIX:')
print(tokenizer.decode(prefix))
print()
print('TRUE NEXT TOKEN:', repr(tokenizer.decode([target_next])))
print()
print('HMM top-k')
display(topk_summary(hmm_next_token_probs(prefix, hmm_model).cpu(), tokenizer, TOPK))
print()
print(f'CHMM top-k ({selected_chmm_ckpt.name})')
display(topk_summary(chmm_next_token_probs(prefix, chmm_model).cpu(), tokenizer, TOPK))


PREFIX:
 Mark Olshansky and Alex Sand

TRUE NEXT TOKEN: 'ro'

HMM top-k


,rank,token_id,token_str,prob
0,1,805,'man',0.035820
1,2,268,'en',0.018510
2,3,6,"""'""",0.015404
3,4,263,'er',0.015173
4,5,1636,'ley',0.014301
5,6,259,'in',0.011516
6,7,261,'on',0.010881
7,8,88,'y',0.010645
8,9,293,'le',0.009545
9,10,78,'o',0.009034



CHMM top-k (checkpoint-300)


,rank,token_id,token_str,prob
0,1,805,'man',0.063019
1,2,882,'erson',0.053903
2,3,12135,'storm',0.038825
3,4,22664,'wic',0.029768
4,5,3900,'berg',0.025863
5,6,282,'al',0.023548
6,7,263,'er',0.023183
7,8,1754,'ler',0.022824
8,9,874,'als',0.022470
9,10,72,'i',0.022470


In [8]:
generation_rows = []
base_prefixes = []
full_text = []
for idx in range(10):
    seq = usable[idx]
    prefix = seq[:PREFIX_LEN].tolist()
    base_prefixes.append(prefix)
    full_text.append(seq.tolist())

for idx, prefix in enumerate(base_prefixes):
    hmm_tokens = sample_autoregressive(hmm_next_token_probs, hmm_model, prefix, GEN_MAX_NEW_TOKENS, EOS_TOKEN_ID, SEED + idx)
    chmm_tokens = sample_autoregressive(chmm_next_token_probs, chmm_model, prefix, GEN_MAX_NEW_TOKENS, EOS_TOKEN_ID, SEED + idx)
    generation_rows.append({
        'example_id': idx,
        'prefix': tokenizer.decode(prefix),
        'hmm_sample': tokenizer.decode(hmm_tokens),
        'chmm_sample': tokenizer.decode(chmm_tokens),
        'full_text': tokenizer.decode(full_text[idx])
    })

pd.DataFrame(generation_rows)


,example_id,prefix,hmm_sample,chmm_sample,full_text
0,0,Mark Olshansky and Alex Sand,Mark Olshansky and Alex Sanderd Address: Try ...,Mark Olshansky and Alex Sandhya dissolution p...,Mark Olshansky and Alex Sandro have quickly e...
1,1,하고 말,하고 말 욭이 걭의 쐨됛작다 걀 결 �,하고 말��u.ieman) �ʐ�・ルデュ ブ� � сг�ขง,하고 말어요? ('I can't help wondering if my dad has...
2,2,• • •,• • • G 2016 at 8-09.12 - Charite | | | | ...,• • • G on Sunday (`\n\nη・・ルキはない だ小‍�� 해,• • • • • • • • • • • • • • • • • • • • • ...
3,3,***************__ *************** //__________...,***************__ *************** //__________...,***************__ *************** //__________...,***************__ *************** //__________...
4,4,༼ つ ◕_�,"༼ つ ◕_◕ ༽つ bad 《Sang"" and ""to said, tendingand...",༼ つ ◕_◕ ̆͝ウゼ・�لـ th ek}} Tand Adjects like her...,༼ つ ◕_◕ ༽つ ༼ つ ◕_◕ ༽つ ༼
5,5,______________________________________________...,______________________________________________...,______________________________________________...,______________________________________________...
6,6,�任玉的�,�任玉的敏郴の匫泏ガルキス �腋�龍装・�生 峫龍喚士レ�機�,�任玉的教อ(̫�界( \ / Houdini. 2) 147 ...,�任玉的数难颊数得做了。如果要�
7,7,iay 0 yololo 10,iay 0 yololo 107.000000\n\n\nDisc Kingdom E 1...,iay 0 yololo 10\n\n\n\nStart with a few momen...,iay 0 yololo 10 ws 0 chungew 0 xie 0 eh 0 vy ...
8,8,an alarming study has taken place.,an alarming study has taken place. The differ...,an alarming study has taken place. The differ...,an alarming study has taken place. The invest...
9,9,______________________________________________...,______________________________________________...,______________________________________________...,______________________________________________...


### How to read this notebook

- `chmm_metrics_df` shows how each saved CHMM checkpoint scores on the same held-out dev subset. Lower `nll_per_token_nats` is better.
- `comparison_df` compares the HMM against the selected CHMM checkpoint.
- `top10_overlap` and `js_divergence` tell you whether HMM and CHMM make similar next-token predictions on the same prefixes.
- The sample table is qualitative only, but it helps spot repetitive collapse, bad tokenization, or too-early EOS.
